# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip -q install duckdb huggingface_hub


In [7]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [8]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 100
""").df()

features.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,<NA>


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

### gsc_impressions
- Meaning: Number of times the page appeared in Google Search.
- Missing values: Missing values are kept as NULL.
- Available when: Yes, available before prediction.

### gsc_clicks
- Meaning: Number of clicks from Google Search.
- Missing values: Missing values are kept as zero.
- Available when: Yes.

### gsc_avg_position
- Meaning: Average ranking position in Google Search.
- Missing values: Missing values are kept as zero.
- Available when: Yes.

### gsc_sum_position
- Meaning: Sum of ranking positions.
- Missing values: Missing values are kept as zero
- Available when: Yes.

### scroll_events
- Meaning: User scroll interactions.
- Missing values: Missing values are kept as zero.
- Available when: Yes.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [10]:
leakage_check = con.sql(f"""
SELECT
    report_date,
    gsc_clicks,
    gsc_impressions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 10
""").df()

leakage_check

,report_date,gsc_clicks,gsc_impressions
0,2026-03-01,0,20
1,2026-03-01,0,1
2,2026-03-01,1,125
3,2026-03-01,0,7
4,2026-03-01,0,11
5,2026-03-01,1,239
6,2026-03-01,0,191
7,2026-03-01,0,55
8,2026-03-01,0,77
9,2026-03-01,0,2


## Leakage Check

No future information or label-derived columns were used.
Only daily metrics available at prediction time were selected.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Excluded Fields

- client_has_gsc: Availability flag, not a predictive feature.
- client_has_ga4: Availability flag.
- gsc_data_available: Indicates data availability only.
- ga4_data_available: Indicates data availability only.
- client_hash_id: Identifier only.
- content_hash_id: Identifier only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.